# DL_01: Data Preparation & Preprocessing Pipeline
**Causal NLP Reasoning — Phase 2: Deep Learning**

---

## 1. Purpose and Scope
This notebook establishes reproducible data preparation and preprocessing pipelines for both Phase 2 benchmark streams:
1. **Corr2Cause**: Abstract causal inference formulated as Natural Language Inference (NLI) premise-hypothesis entailment pairs.
2. **BioCause**: Candidate-conditioned directed causal relation classification using annotated argument spans from open-access PMC biomedical articles.

Key requirements:
- Prepare Corr2Cause and BioCause datasets independently (strict boundary enforcement).
- Document dataset schemas, label distributions, and directional label semantics.
- Ingest canonical input files from `data/raw/` and `data/processed/`.
- Preserve the canonical Corr2Cause split counts: Train (205,734), Validation (1,076), and Test (1,162).
- Maintain the 12-train / 4-validation / 4-test PMC article-level split isolation for BioCause to prevent cross-article data leakage.
- Construct explicit argument boundary markers (`[E1]...[/E1]` and `[E2]...[/E2]`) preserving original argument order and contextual text.
- Define expected model input tensors and target outputs for transformer fine-tuning in `DL_02` and `DL_03`.


## 2. Experimental Methodology & Theoretical Background

### Causal Relation Classification vs. Causal Entailment
- **Corr2Cause (Symbolic NLI)**: Evaluates whether language models deduce valid causal graph relations solely from statistical independence statements:
  - **Premise**: Verbalized statistical independence constraints among variables.
  - **Hypothesis**: Directed causal conjecture (e.g. "*A causes B.*")
  - **Target Label**: `1` (Entailed) or `0` (Refuted).
  - **Canonical Split Distribution**: Train = 205,734; Validation = 1,076; Test = 1,162. Validation must remain strictly separate from training.

- **BioCause (Biomedical Candidate-Conditioned Relation Classification)**: Given co-occurring biomedical entity spans ($Arg_1$, $Arg_2$) in scientific text, classify whether the directed relationship $Arg_1 \to Arg_2$ holds:
  - **Label Semantics**: Label `1` denotes that $Arg_1$ causes $Arg_2$. Label `0` denotes that $Arg_1$ does not represent the gold directed relation $Arg_1 \to Arg_2$ under the candidate-generation convention (which includes reversed directional pairs $(Arg_2, Arg_1)$ as well as non-causal pairs). Label `0` must not be mischaracterized as universally non-causal.
  - **Class Imbalance**: Candidate pairing produces 10,086 training pairs (621 positive, 9,465 negative), an actual negative-to-positive ratio of **15.24:1** (Validation: 131 pos / 1,783 neg = 13.61:1; Test: 100 pos / 1,308 neg = 13.08:1).

### Article-Level Split Isolation (Zero Data Leakage)
BioCause strictly partitions the 20 PMC publications by article ID:
- **Train Split**: 12 PMC Articles (621 gold causal relations, 10,086 candidate pairs)
- **Validation Split**: 4 PMC Articles (131 gold causal relations, 1,914 candidate pairs)
- **Test Split**: 4 PMC Articles (100 gold causal relations, 1,408 candidate pairs)


## 3. Expected Inputs

| Stream | File Path | Canonical Size / Dimensions | Key Fields |
| :--- | :--- | :--- | :--- |
| **Corr2Cause Train** | `data/raw/train.csv` | 205,734 rows | `input, label, num_variables, template` |
| **Corr2Cause Val** | `data/raw/validation.csv` | 1,076 rows | `input, label, num_variables, template` |
| **Corr2Cause Test** | `data/raw/test.csv` | 1,162 rows | `input, label, num_variables, template` |
| **BioCause Candidates** | `data/processed/biocause_candidates.csv` | 13,408 candidate pairs | `candidate_id, article_id, section_id, arg1_text, arg2_text, between_text, label, split...` |
| **BioCause Gold Relations** | `data/processed/biocause_relations.csv` | 852 gold relations | `relation_id, article_id, cause_id, effect_id...` |


## 4. Technical Responsibilities & Implementation Requirements

1. **Environment & Portable Root Discovery**: Locate repo root dynamically and load `dl_utils`.
2. **Corr2Cause Split Ingestion**: Verify exact row counts (205,734 train, 1,076 val, 1,162 test) and verify binary label balance.
3. **BioCause Article-Level Isolation Audit**: Check that article IDs across splits are strictly disjoint.
4. **Argument Marker Construction**: Format BioCause inputs with special tokens `[E1]`, `[/E1]`, `[E2]`, `[/E2]`. Ensure markers reference original text offsets, preserve original text order, include surrounding context, and check for span validity/overlap.
5. **Dynamic Class Weight Calculation**: Compute $w_{\text{pos}} = N_{\text{neg}} / N_{\text{pos}}$ strictly on the training split ($9,465 / 621 \approx 15.24$). Never leak validation or test counts into weight computation.
6. **Token Length Profiling**: Profile sequence lengths under `distilbert-base-uncased` and `microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext` tokenizers.
7. **Dataset Serialization**: Export clean model-ready splits to `data/processed/`.


---

## Section A — Environment Setup
Set up repository imports, configure reproducibility, detect compute device, and display environment information.

In [ ]:
# Section A: Environment Setup & Portable Imports
import sys
import os
import warnings
from pathlib import Path

# ---- Repository root discovery ----
def find_repo_root():
    """Locate repository root across local, Colab, and Kaggle."""
    curr = Path(os.getcwd()).resolve()
    for p in [curr] + list(curr.parents):
        if (p / 'AML' / 'causal_extractors.py').exists() and (p / 'DL' / 'dl_utils.py').exists():
            return p
    return curr

ROOT_DIR = find_repo_root()
for p in [ROOT_DIR, ROOT_DIR / 'AML', ROOT_DIR / 'DL']:
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

import dl_utils
import pandas as pd
import numpy as np

print(f'Repository root: {ROOT_DIR}')
print(f'Python version:  {sys.version}')
print(f'pandas version:  {pd.__version__}')
print(f'numpy version:   {np.__version__}')

In [ ]:
# Reproducibility & device detection
dl_utils.set_seed(42)
DEVICE = dl_utils.get_device()
print(f'Compute device:  {DEVICE}')

try:
    import torch
    print(f'PyTorch version: {torch.__version__}')
    if torch.cuda.is_available():
        print(f'CUDA device:     {torch.cuda.get_device_name(0)}')
except ImportError:
    print('PyTorch: not installed')

try:
    import transformers
    print(f'Transformers:    {transformers.__version__}')
except ImportError:
    print('Transformers: not installed')

---

## Section B — Corr2Cause Data Preparation
Load canonical Corr2Cause CSV splits and verify row counts, columns, labels, missing values, and duplicates.

In [ ]:
# Section B: Corr2Cause Data Ingestion & Split Verification

CORR2CAUSE_RAW_DIR = ROOT_DIR / 'data' / 'raw'

c2c_files = {
    'train': CORR2CAUSE_RAW_DIR / 'train.csv',
    'validation': CORR2CAUSE_RAW_DIR / 'validation.csv',
    'test': CORR2CAUSE_RAW_DIR / 'test.csv',
}

# Check that all files exist before proceeding
missing_c2c = [name for name, path in c2c_files.items() if not path.exists()]
if missing_c2c:
    msg = (
        f'MISSING Corr2Cause raw files: {missing_c2c}\n'
        f'Expected location: {CORR2CAUSE_RAW_DIR}\n'
        f'Required files: train.csv, validation.csv, test.csv\n'
        f'Download from HuggingFace: https://huggingface.co/datasets/causalnlp/corr2cause\n'
        f'Or run 00_setup.ipynb to automatically stage the datasets.'
    )
    raise FileNotFoundError(msg)

In [ ]:
# Load Corr2Cause splits
df_c2c_train = pd.read_csv(c2c_files['train'])
df_c2c_val = pd.read_csv(c2c_files['validation'])
df_c2c_test = pd.read_csv(c2c_files['test'])

print('=== Corr2Cause Split Sizes ===')
print(f'  Train:      {len(df_c2c_train):>10,} (expected {dl_utils.CORR2CAUSE_SPLIT_COUNTS["train"]:,})')
print(f'  Validation: {len(df_c2c_val):>10,} (expected {dl_utils.CORR2CAUSE_SPLIT_COUNTS["validation"]:,})')
print(f'  Test:       {len(df_c2c_test):>10,} (expected {dl_utils.CORR2CAUSE_SPLIT_COUNTS["test"]:,})')

# Verify exact row counts
assert len(df_c2c_train) == dl_utils.CORR2CAUSE_SPLIT_COUNTS['train'], \
    f'Train count mismatch: {len(df_c2c_train)} vs {dl_utils.CORR2CAUSE_SPLIT_COUNTS["train"]}'
assert len(df_c2c_val) == dl_utils.CORR2CAUSE_SPLIT_COUNTS['validation'], \
    f'Val count mismatch: {len(df_c2c_val)} vs {dl_utils.CORR2CAUSE_SPLIT_COUNTS["validation"]}'
assert len(df_c2c_test) == dl_utils.CORR2CAUSE_SPLIT_COUNTS['test'], \
    f'Test count mismatch: {len(df_c2c_test)} vs {dl_utils.CORR2CAUSE_SPLIT_COUNTS["test"]}'
print('\n✓ All Corr2Cause split counts verified.')

In [ ]:
# Validate required columns
C2C_REQUIRED_COLS = ['input', 'label']
for split_name, df in [('train', df_c2c_train), ('val', df_c2c_val), ('test', df_c2c_test)]:
    dl_utils.validate_dataset_schema(df, C2C_REQUIRED_COLS, f'Corr2Cause-{split_name}')
print('✓ All required Corr2Cause columns present.')

# Verify label format (binary: 0 or 1)
for split_name, df in [('train', df_c2c_train), ('val', df_c2c_val), ('test', df_c2c_test)]:
    unique_labels = set(df['label'].dropna().unique())
    assert unique_labels.issubset({0, 1}), f'{split_name}: non-binary labels found: {unique_labels}'
print('✓ All Corr2Cause labels are binary (0/1).')

# Check for missing values in critical fields
for split_name, df in [('train', df_c2c_train), ('val', df_c2c_val), ('test', df_c2c_test)]:
    null_input = df['input'].isna().sum()
    null_label = df['label'].isna().sum()
    assert null_input == 0, f'{split_name}: {null_input} null inputs'
    assert null_label == 0, f'{split_name}: {null_label} null labels'
print('✓ No missing values in input or label columns.')

# Label distribution
print('\n=== Corr2Cause Label Distributions ===')
for split_name, df in [('train', df_c2c_train), ('validation', df_c2c_val), ('test', df_c2c_test)]:
    vc = df['label'].value_counts().sort_index()
    print(f'  {split_name:>12}: label_0={vc.get(0, 0):>7,}, label_1={vc.get(1, 0):>7,}')

---

## Section C — BioCause Data Preparation
Load BioCause candidate pairs, verify article-level split isolation, candidate identifiers,
argument text/offsets, binary labels, and expected counts.

In [ ]:
# Section C: BioCause Data Ingestion & Article-Level Split Isolation Audit

BIOCAUSE_CANDS_PATH = ROOT_DIR / 'data' / 'processed' / 'biocause_candidates.csv'

if not BIOCAUSE_CANDS_PATH.exists():
    msg = (
        f'MISSING BioCause candidates file: {BIOCAUSE_CANDS_PATH}\n'
        f'This file is generated by 00_setup.ipynb and AML/AML_01_data_features.ipynb.\n'
        f'Run those notebooks first to produce the candidate pairs.'
    )
    raise FileNotFoundError(msg)

df_bio = pd.read_csv(BIOCAUSE_CANDS_PATH)
print(f'BioCause candidates loaded: {len(df_bio):,} rows')
print(f'Columns: {list(df_bio.columns)}')

In [ ]:
# Validate BioCause candidates
bio_summary = dl_utils.validate_biocause_candidates(df_bio, 'BioCause Candidates')

print('\n=== BioCause Split Summary ===')
print(f'{"Split":<12} {"Articles":>10} {"Candidates":>12} {"Positive":>10} {"Negative":>10}')
print('-' * 58)
for split_name in ['train', 'validation', 'test']:
    s = bio_summary[split_name]
    print(f'{split_name:<12} {s["articles"]:>10} {s["total"]:>12,} {s["positive"]:>10,} {s["negative"]:>10,}')

print('\n✓ BioCause candidate validation passed (article isolation, labels, IDs).')

In [ ]:
# Verify article-level split isolation explicitly
train_articles = set(df_bio[df_bio['split'] == 'train']['article_id'].unique())
val_articles = set(df_bio[df_bio['split'] == 'validation']['article_id'].unique())
test_articles = set(df_bio[df_bio['split'] == 'test']['article_id'].unique())

assert len(train_articles & val_articles) == 0, 'Train/Val article overlap detected!'
assert len(train_articles & test_articles) == 0, 'Train/Test article overlap detected!'
assert len(val_articles & test_articles) == 0, 'Val/Test article overlap detected!'

print(f'Train articles ({len(train_articles)}):  {sorted(train_articles)}')
print(f'Val articles ({len(val_articles)}):    {sorted(val_articles)}')
print(f'Test articles ({len(test_articles)}):   {sorted(test_articles)}')
print('\n✓ Zero cross-split article leakage confirmed.')

In [ ]:
# Verify expected split counts against canonical values
expected = dl_utils.BIOCAUSE_CANDIDATE_COUNTS

for split_name in ['train', 'validation', 'test']:
    actual = bio_summary[split_name]
    exp = expected[split_name]
    assert actual['total'] == exp['total'], \
        f'{split_name} total: {actual["total"]} vs expected {exp["total"]}'
    assert actual['positive'] == exp['positive'], \
        f'{split_name} positive: {actual["positive"]} vs expected {exp["positive"]}'
    assert actual['negative'] == exp['negative'], \
        f'{split_name} negative: {actual["negative"]} vs expected {exp["negative"]}'

print('✓ All BioCause candidate counts match expected values.')

In [ ]:
# Check for duplicate candidate IDs
dup_count = df_bio['candidate_id'].duplicated().sum()
assert dup_count == 0, f'Found {dup_count} duplicate candidate_ids'
print(f'✓ No duplicate candidate_ids ({len(df_bio):,} unique candidates).')

# Verify argument identifiers and text are present
for col in ['arg1_id', 'arg2_id', 'arg1_text', 'arg2_text']:
    null_count = df_bio[col].isna().sum()
    print(f'  {col}: {null_count} nulls')
    assert null_count == 0, f'{col} has {null_count} null values'

---

## Section D — Entity Marker Construction
Insert `[E1]...[/E1]` and `[E2]...[/E2]` entity boundary markers using character offsets.
Verify that Arg1 and Arg2 identities are maintained and demonstrate handling of different argument orders.

In [ ]:
# Section D: Entity Marker Construction via Source Section Reconstruction
# Check required candidate columns
required_cand_cols = ["arg1_start", "arg1_end", "arg2_start", "arg2_end", "arg1_text", "arg2_text", "section_id"]
missing_cand_cols = [c for c in required_cand_cols if c not in df_bio.columns]
if missing_cand_cols:
    raise ValueError(f"BioCause candidates missing required offset/span columns: {missing_cand_cols}")

# Locate and cache raw section .txt files from data/raw/biocause/BioCause_corpus/ (or fallback data/raw/biocause/)
unique_sec_ids = df_bio["section_id"].unique()
print(f"Loading raw section files for {len(unique_sec_ids)} unique BioCause sections...")

sec_text_cache = {}
missing_sec_files = []

for sec_id in unique_sec_ids:
    try:
        sec_text = dl_utils.load_biocause_section_text(sec_id)
        sec_text_cache[sec_id] = sec_text
    except FileNotFoundError:
        missing_sec_files.append(sec_id)

if missing_sec_files:
    raise FileNotFoundError(
        f"Missing {len(missing_sec_files)} BioCause raw section .txt files in "
        f"data/raw/biocause/BioCause_corpus/ or data/raw/biocause/. "
        f"Sample missing sections: {missing_sec_files[:5]}. "
        f"Run 00_setup.ipynb or ensure the BioCause corpus is staged."
    )

print(f"✓ Successfully loaded {len(sec_text_cache)} BioCause section text files.")

In [ ]:
# Demonstrate marker insertion and span validation on sample candidates
print("=== Entity Marker Construction Examples ===")
sample_rows = df_bio.head(5)

for idx, row in sample_rows.iterrows():
    sec_text = sec_text_cache[row["section_id"]]
    marked = dl_utils.reconstruct_biocause_context_with_markers(
        section_text=sec_text,
        arg1_start=int(row["arg1_start"]),
        arg1_end=int(row["arg1_end"]),
        arg2_start=int(row["arg2_start"]),
        arg2_end=int(row["arg2_end"]),
        arg1_text=str(row["arg1_text"]),
        arg2_text=str(row["arg2_text"]),
        candidate_id=str(row["candidate_id"]),
        max_length=dl_utils.DEFAULT_BIOCAUSE_MAX_LENGTH,  # 384 tokens
    )
    
    # Assert markers are present
    assert "[E1]" in marked and "[/E1]" in marked, "Missing E1 markers"
    assert "[E2]" in marked and "[/E2]" in marked, "Missing E2 markers"
    
    print(f"\n[{idx}] candidate_id={row['candidate_id']}")
    print(f"  Arg1: {row['arg1_text']!r} [{row['arg1_start']}:{row['arg1_end']}]")
    print(f"  Arg2: {row['arg2_text']!r} [{row['arg2_start']}:{row['arg2_end']}]")
    print(f"  Marked & Cropped: {marked[:200]}..." if len(marked) > 200 else f"  Marked: {marked}")

In [ ]:
# Reconstruct marked source context for all BioCause candidates
print("Reconstructing marked source context for all candidates...")
marked_texts = []
reconstruction_errors = []

for idx, row in df_bio.iterrows():
    sec_text = sec_text_cache[row["section_id"]]
    try:
        marked = dl_utils.reconstruct_biocause_context_with_markers(
            section_text=sec_text,
            arg1_start=int(row["arg1_start"]),
            arg1_end=int(row["arg1_end"]),
            arg2_start=int(row["arg2_start"]),
            arg2_end=int(row["arg2_end"]),
            arg1_text=str(row["arg1_text"]),
            arg2_text=str(row["arg2_text"]),
            candidate_id=str(row["candidate_id"]),
            max_length=dl_utils.DEFAULT_BIOCAUSE_MAX_LENGTH,  # 384 tokens
        )
        marked_texts.append(marked)
    except Exception as e:
        reconstruction_errors.append((row["candidate_id"], str(e)))

if reconstruction_errors:
    raise ValueError(
        f"Encountered {len(reconstruction_errors)} errors during BioCause context reconstruction. "
        f"First error: {reconstruction_errors[0]}"
    )

df_bio["marked_text"] = marked_texts
print(f"✓ Reconstructed marked and cropped source context for all {len(df_bio):,} BioCause candidates within 384 tokens.")

In [ ]:
# Demonstrate different argument orders
print('=== Argument Order Verification ===')
if HAS_OFFSETS:
    # Find cases where arg1 appears after arg2 in text
    arg2_first = df_bio[df_bio['arg2_start'] < df_bio['arg1_start']]
    arg1_first = df_bio[df_bio['arg1_start'] < df_bio['arg2_start']]
    
    print(f'  Arg1 before Arg2: {len(arg1_first):,} candidates')
    print(f'  Arg2 before Arg1: {len(arg2_first):,} candidates')
    
    # Show one example of each ordering
    if len(arg1_first) > 0:
        row = arg1_first.iloc[0]
        print(f'\nArg1-first example (candidate {row["candidate_id"]}):')
        print(f'  Arg1=[{row["arg1_start"]}:{row["arg1_end"]}] "{row["arg1_text"]}"')
        print(f'  Arg2=[{row["arg2_start"]}:{row["arg2_end"]}] "{row["arg2_text"]}"')
        print(f'  Marked: {row["marked_text"][:200]}')
    
    if len(arg2_first) > 0:
        row = arg2_first.iloc[0]
        print(f'\nArg2-first example (candidate {row["candidate_id"]}):')
        print(f'  Arg1=[{row["arg1_start"]}:{row["arg1_end"]}] "{row["arg1_text"]}"')
        print(f'  Arg2=[{row["arg2_start"]}:{row["arg2_end"]}] "{row["arg2_text"]}"')
        print(f'  Marked: {row["marked_text"][:200]}')
else:
    print('  (Offset columns not available; argument order demo skipped.)')

---

## Section E — Tokenizer Profiling
Load tokenizers, register entity markers as special tokens, profile token lengths,
calculate truncation statistics, and apply context cropping where necessary.

In [ ]:
# Section E: Tokenizer Profiling (DistilBERT & PubMedBERT)
# Profile sequence lengths across both datasets to document sensible sequence length limits.

try:
    from transformers import AutoTokenizer
    HAS_TRANSFORMERS = True
except ImportError:
    HAS_TRANSFORMERS = False
    print("WARNING: transformers not installed. Skipping tokenizer profiling.")
    print("Install with: pip install -r requirements-dl.txt")

In [ ]:
if HAS_TRANSFORMERS:
    # 1. Profile DistilBERT on Corr2Cause inputs
    print("=== 1. Corr2Cause Tokenizer Profiling (DistilBERT) ===")
    distilbert_tokenizer = AutoTokenizer.from_pretrained(dl_utils.DISTILBERT_CHECKPOINT)
    
    # Format sample Corr2Cause inputs
    sample_c2c_texts = [
        f"[CLS] {row['input']} [SEP]"
        for _, row in df_c2c_train.head(1000).iterrows()
    ]
    
    c2c_profile = dl_utils.profile_token_lengths(
        texts=sample_c2c_texts,
        tokenizer=distilbert_tokenizer,
        max_samples=1000,
        target_budget=dl_utils.DEFAULT_CORR2CAUSE_MAX_LENGTH,  # 256 tokens
    )
    
    print(f"  Samples: {c2c_profile['count']:,}")
    print(f"  Min:     {c2c_profile['min']} tokens")
    print(f"  Mean:    {c2c_profile['mean']} tokens")
    print(f"  Median:  {c2c_profile['median']:.0f} tokens")
    print(f"  P95:     {c2c_profile['p95']:.0f} tokens")
    print(f"  P99:     {c2c_profile['p99']:.0f} tokens")
    print(f"  Max:     {c2c_profile['max']} tokens")
    print(f"  Within {dl_utils.DEFAULT_CORR2CAUSE_MAX_LENGTH} budget: {c2c_profile['pct_within_budget']:.1f}%")
    print(f"\n✓ Documented setting for Corr2Cause: MAX_LENGTH = {dl_utils.DEFAULT_CORR2CAUSE_MAX_LENGTH} tokens.")

In [ ]:
if HAS_TRANSFORMERS:
    # 2. Profile PubMedBERT on marked BioCause inputs
    print("=== 2. BioCause Tokenizer Profiling (PubMedBERT) ===")
    pubmedbert_tokenizer = dl_utils.prepare_tokenizer_and_inputs(
        checkpoint_name=dl_utils.PUBMEDBERT_CHECKPOINT,
        max_length=dl_utils.DEFAULT_BIOCAUSE_MAX_LENGTH,
        additional_special_tokens=dl_utils.ENTITY_MARKERS,
    )
    print(f"  Vocab size with markers: {len(pubmedbert_tokenizer)}")
    
    # Verify entity markers are recognized tokens
    for marker in dl_utils.ENTITY_MARKERS:
        token_id = pubmedbert_tokenizer.convert_tokens_to_ids(marker)
        assert token_id != pubmedbert_tokenizer.unk_token_id, f"{marker} not recognized!"
    
    bio_profile = dl_utils.profile_token_lengths(
        texts=df_bio["marked_text"].head(1000).tolist(),
        tokenizer=pubmedbert_tokenizer,
        max_samples=1000,
        target_budget=dl_utils.DEFAULT_BIOCAUSE_MAX_LENGTH,  # 384 tokens
    )
    
    print(f"  Samples: {bio_profile['count']:,}")
    print(f"  Min:     {bio_profile['min']} tokens")
    print(f"  Mean:    {bio_profile['mean']} tokens")
    print(f"  Median:  {bio_profile['median']:.0f} tokens")
    print(f"  P95:     {bio_profile['p95']:.0f} tokens")
    print(f"  P99:     {bio_profile['p99']:.0f} tokens")
    print(f"  Max:     {bio_profile['max']} tokens")
    print(f"  Within {dl_utils.DEFAULT_BIOCAUSE_MAX_LENGTH} budget: {bio_profile['pct_within_budget']:.1f}%")
    print(f"\n✓ Documented setting for BioCause: MAX_LENGTH = {dl_utils.DEFAULT_BIOCAUSE_MAX_LENGTH} tokens.")

---

## Section F — Class Imbalance
Calculate the BioCause positive class weight using the training split only.

In [ ]:
# Section F: Dynamic Training Class Weight Calculation

df_bio_train = df_bio[df_bio['split'] == 'train']
n_pos_train = int((df_bio_train['label'] == 1).sum())
n_neg_train = int((df_bio_train['label'] == 0).sum())

pos_weight = dl_utils.calculate_dynamic_class_weight(
    n_negative=n_neg_train,
    n_positive=n_pos_train,
)

print('=== BioCause Dynamic Class Weight ===')
print(f'  Training positive: {n_pos_train:,}')
print(f'  Training negative: {n_neg_train:,}')
print(f'  pos_weight = {n_neg_train} / {n_pos_train} = {pos_weight:.4f}')
print(f'  Expected: ~15.2415')
print(f'\n✓ Class weight computed strictly from training split.')
print(f'  NOTE: Validation ({dl_utils.BIOCAUSE_CANDIDATE_COUNTS["validation"]["positive"]} pos) '
      f'and test ({dl_utils.BIOCAUSE_CANDIDATE_COUNTS["test"]["positive"]} pos) '
      f'labels NOT used in weight calculation.')

---

## Section G — Export Processed Datasets
Export six model-ready CSV files to `data/processed/`.

In [ ]:
# Section G: Export Model-Ready Datasets

PROCESSED_DIR = ROOT_DIR / 'data' / 'processed'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# --- Corr2Cause exports ---
# Add split column for downstream clarity
df_c2c_train_out = df_c2c_train.copy()
df_c2c_train_out['split'] = 'train'
df_c2c_val_out = df_c2c_val.copy()
df_c2c_val_out['split'] = 'validation'
df_c2c_test_out = df_c2c_test.copy()
df_c2c_test_out['split'] = 'test'

c2c_exports = {
    'dl_corr2cause_train.csv': df_c2c_train_out,
    'dl_corr2cause_val.csv': df_c2c_val_out,
    'dl_corr2cause_test.csv': df_c2c_test_out,
}

print('=== Exporting Corr2Cause Processed Datasets ===')
for fname, df in c2c_exports.items():
    path = str(PROCESSED_DIR / fname)
    dl_utils.export_predictions(df, path, format_type='csv')
    print(f'  ✓ {fname}: {len(df):>10,} rows')

In [ ]:
# --- BioCause exports ---
# Select columns needed by downstream DL notebooks
biocause_export_cols = [
    col for col in [
        'candidate_id', 'article_id', 'section_id',
        'arg1_id', 'arg2_id', 'arg1_text', 'arg2_text',
        'arg1_start', 'arg1_end', 'arg2_start', 'arg2_end',
        'between_text', 'section_text', 'marked_text',
        'label', 'split', 'arg1_before_arg2',
    ] if col in df_bio.columns
]

bio_exports = {
    'dl_biocause_train.csv': df_bio[df_bio['split'] == 'train'][biocause_export_cols],
    'dl_biocause_val.csv': df_bio[df_bio['split'] == 'validation'][biocause_export_cols],
    'dl_biocause_test.csv': df_bio[df_bio['split'] == 'test'][biocause_export_cols],
}

print('\n=== Exporting BioCause Processed Datasets ===')
for fname, df in bio_exports.items():
    path = str(PROCESSED_DIR / fname)
    dl_utils.export_predictions(df, path, format_type='csv')
    print(f'  ✓ {fname}: {len(df):>10,} rows')

In [ ]:
# Final verification: check all 6 export files exist
expected_exports = [
    'dl_corr2cause_train.csv',
    'dl_corr2cause_val.csv',
    'dl_corr2cause_test.csv',
    'dl_biocause_train.csv',
    'dl_biocause_val.csv',
    'dl_biocause_test.csv',
]

print('\n=== Export Verification ===')
all_ok = True
for fname in expected_exports:
    fpath = PROCESSED_DIR / fname
    if fpath.exists():
        size_mb = fpath.stat().st_size / (1024 * 1024)
        print(f'  ✓ {fname} ({size_mb:.2f} MB)')
    else:
        print(f'  ✗ {fname} MISSING')
        all_ok = False

if all_ok:
    print('\n✓ All 6 processed datasets exported successfully.')
else:
    print('\n✗ Some exports missing — check errors above.')

## 5. Expected Outputs & Evaluation Criteria

- `data/processed/dl_corr2cause_{train,val,test}.csv`: Cleaned NLI splits conforming to canonical split counts.
- `data/processed/dl_biocause_{train,val,test}.csv`: Marked relation candidate datasets with preserved 12/4/4 article grouping.
- **Evaluation Criteria**:
  - 100% split integrity: Zero overlap between article IDs in train, val, and test.
  - Argument marker completeness: Both entity spans intact post-formatting and tokenization.
  - Class weight verification: Dynamic training ratio $w_{\text{pos}} \approx 15.24$ computed strictly from training data.
